## Installing Required Libraries

In [2]:
!pip install tensorflow
!pip install numpy
!pip install pandas
!pip install scikit-learn
!pip install matplotlib

## Import Libraries

In [3]:
import tensorflow as tf
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

##Load Dataset from kaggle

In [5]:
df = pd.read_csv("/content/movie reviews.csv")

In [6]:
print("Original shape:",df.shape)

Original shape: (20, 2)


##Validate Columns

In [7]:
print(df.columns)

Index(['review', 'sentiment'], dtype='object')


In [8]:
required_columns = ['review', 'sentiment']
for col in required_columns:
    if col not in df.columns:
        raise Exception(f"Missing column:{col}")

##Clean Sentiment Labels

In [9]:
df['sentiment'] = (
    df['sentiment']
    .astype(str)
    .str.lower()
    .str.strip()
)
df['sentiment'] = df['sentiment'].map({
    'positive':1,
    'negative':0
})
df = df.dropna(subset=['sentiment'])
print("After label mapping:",df.shape)

After label mapping: (14, 2)


##Test Cleaning Function

In [10]:
def clean_text(text):
  if pd.isna(text):
    return ""
  text = str(text).lower()
  #remove URLS
  text = re.sub(r'http\S+|www\S+', '', text)
  #remove HTML tags
  text = re.sub(r'<.*?>', '', text)
  #keep letters and spaces
  text = re.sub(r'[^a-zA-Z\s]', '', text)
  #remove extra spaces and replace with single space
  text = re.sub(r'\s+', ' ', text)
  text = text.strip()
  return text

##Apply Cleaning

In [11]:
df['clean_review'] = df['review'].apply(clean_text)
#Remove empty reviews
df = df[df['clean_review'].str.strip() != ""]
print("After text cleaning:",df.shape)
if len(df) == 0:
  raise Exception("Dataset became empty after cleaning")

After text cleaning: (14, 3)


##Features and Labels

In [12]:
x = df['clean_review']
y = df['sentiment']
print("Sample:",len(x))

Sample: 14


##Train-Test Split

In [13]:
x_train , x_test ,y_train,y_test = train_test_split(
    x,y,test_size=0.2,random_state=42 , stratify=y
)
print("Train Size:",len(x_train))
print("Test Size:",len(x_test))


Train Size: 11
Test Size: 3


##Text Vectorization

In [14]:
vocab_size = 10000
sequence_length = 250
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode = 'int',
    output_sequence_length=sequence_length
)
vectorizer.adapt(x_train)

##Build Model

In [15]:
model = tf.keras.Sequential([
    vectorizer,
    tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=128),
        tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64,activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1,activation='sigmoid')
])

##Compile

In [16]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

##Train

In [17]:
history = model.fit(
    x_train.to_numpy(),
    y_train.to_numpy(),
    validation_split = 0.2,
    epochs = 5,
    batch_size = 32
)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6250 - loss: 0.6861 - val_accuracy: 0.3333 - val_loss: 0.7142
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - accuracy: 0.6250 - loss: 0.6791 - val_accuracy: 0.3333 - val_loss: 0.7298
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step - accuracy: 0.6250 - loss: 0.6904 - val_accuracy: 0.3333 - val_loss: 0.7405
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step - accuracy: 0.6250 - loss: 0.6588 - val_accuracy: 0.3333 - val_loss: 0.7524
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step - accuracy: 0.6250 - loss: 0.6665 - val_accuracy: 0.3333 - val_loss: 0.7646


##Evaluate

In [18]:
loss, accuracy = model.evaluate(
    x_test.to_numpy(),
    y_test.to_numpy()
)
print("\nTest Accuracy:",accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.3333 - loss: 0.7653

Test Accuracy: 0.3333333432674408


##Save Model

In [19]:
model.save("imdb_text_classifier.keras")
print("Model Saved Successfully")

Model Saved Successfully


##Prediction Code

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("imdb_text_classifier.keras")
while True:
  review = input("\nEnter Review:")
  pred = model.predict(np.array([review]),verbose=0)[0][0]
  if pred > 0.5:
    print("Positive Review")
    print("Confidence:",round(pred*100,2),"%")
  else:
    print("Negative Review")
    print("Confidence:",round((1-pred)*100,2),"%")